# Build a minimal fsp237 + Excel + iMM904-essentials model

**Goal:** produce a clean, glucose-viable *Colletotrichum sublineola* FSP237 GEM by:

1. Re-fetching the fresh **fsp237** model from KBase workspace `169876/251/1` (no prior gap-fill state).
2. Adding **all** reactions from the curated *C. higginsianum* Excel set (`Integration_Higgginsianum_GEM - 4JK.xlsx`) — kept even when not glucose-essential, since they may activate under other carbon sources.
3. Adding **only** the minimum reactions from the iMM904 yeast template required for biomass on GMM with glucose uptake = 5 mmol/gDW/h.
4. Running a final **lower-MS-wins** dedup pass so the published model has no duplicate chemistry.

**Duplicate policy:** check at reaction ID *and* stoichiometry level.
- BiGG-style vs ModelSEED with same chemistry → keep ModelSEED.
- Two ModelSEED reactions with same chemistry → keep the one with the **lower numeric suffix** (e.g., `rxn00902` over `rxn27497`).
- All-non-ModelSEED duplicates → keep first, drop the rest.
- Unique non-ModelSEED IDs (transporters, biomass, exchanges) are left alone.

**Outputs (saved into `fsp237_minimal_glucose/`):**
- `fsp237_initial_snapshot.json` — the pristine KBase fetch
- `fsp237_minimal_glucose.json` — the final model (post-dedup)
- `fsp237_minimal_glucose_fluxes.tsv` — FBA fluxes
- `fsp237_build_log.tsv` — per-reaction add/skip/replace log
- `fsp237_dedup_removed_log.tsv` — every reaction removed by the final dedup pass and its surviving counterpart
- `fsp237_minimal_glucose_flux_map.html` — compartment-coloured Escher map

In [ ]:
import json, re, os, contextlib, io, time, random
import pandas as pd
import cobra
from cobra import Reaction, Metabolite
import cobrakbase
from escher import Builder

random.seed(42)

base_dir  = '/home/janakae/fungalTemplate/imm904CobraModel'
out_dir   = f'{base_dir}/fsp237_minimal_glucose'
xl_path   = f'{base_dir}/Integration_Higgginsianum_GEM - 4JK.xlsx'
imm_path  = f'{base_dir}/COBRA_Verson2_6_iMM904_CC_swapped_4.json'
escher_map_path = f'{base_dir}/iMM904_Central_carbon_metabolism_March28.json'

snap_path  = f'{out_dir}/fsp237_initial_snapshot.json'
final_json = f'{out_dir}/fsp237_minimal_glucose.json'
final_tsv  = f'{out_dir}/fsp237_minimal_glucose_fluxes.tsv'
log_tsv    = f'{out_dir}/fsp237_build_log.tsv'
dedup_tsv  = f'{out_dir}/fsp237_dedup_removed_log.tsv'
map_html   = f'{out_dir}/fsp237_minimal_glucose_flux_map.html'

GLC_UPTAKE       = 5.0
BIO_THRESH_FRAC  = 0.95
EPS_BIO          = 1e-6

os.makedirs(out_dir, exist_ok=True)
build_log = []

## Step 1 — Fetch fresh fsp237 from KBase

Pulls the *C. sublineola* FSP237 model from KBase workspace `169876/251/1` and saves a snapshot so the rest of the notebook is deterministic on re-run.

In [ ]:
kbase = cobrakbase.KBaseAPI('4IJU2EOS5G2O5N64QBTIURKE3PZF2JTE')
fsp237 = kbase.get_from_ws('169876/251/1')
print(f'fsp237 baseline: {len(fsp237.reactions)} rxns, {len(fsp237.metabolites)} mets, {len(fsp237.genes)} genes')
cobra.io.save_json_model(fsp237, snap_path)
print(f'snapshot saved: {snap_path}')

# Reload from snapshot for deterministic downstream behavior
fsp237 = cobra.io.load_json_model(snap_path)

## Step 2 — Load Excel reactions + iMM904 template

The Excel file has 8 sheets (AA/Nuc/Vit/Lipids × basic/Net), each with a `rxn id` column and an `equation` column. We collect all unique reaction IDs across all sheets.

The iMM904 template provides reaction objects (stoichiometry + metabolites) for the Excel IDs whenever possible.

In [ ]:
imm904 = cobra.io.load_json_model(imm_path)
print(f'iMM904: {len(imm904.reactions)} rxns, {len(imm904.metabolites)} mets')

xls = pd.ExcelFile(xl_path)
excel_rows = []
for s in xls.sheet_names:
    df = pd.read_excel(xl_path, sheet_name=s)
    if 'rxn id' not in df.columns: continue
    keep = [c for c in ['rxn id','compartment','equation','name (ModelSEED)','bigg id','direction'] if c in df.columns]
    sub = df[keep].copy()
    sub['source_sheet'] = s
    excel_rows.append(sub)
excel_df = pd.concat(excel_rows, ignore_index=True)
excel_df['rxn id'] = excel_df['rxn id'].astype(str).str.strip().str.replace(' ', '', regex=False)
excel_df = excel_df[(excel_df['rxn id'] != '') & (excel_df['rxn id'] != 'nan')]
excel_df = excel_df.drop_duplicates(subset=['rxn id'])
print(f'Excel reactions (unique ids): {len(excel_df)}')

## Step 3 — Helpers: duplicate detection + equation parser

- `canon_stoich(rxn)` → hashable tuple of `(metabolite_id, coefficient)` pairs, ignoring H+ (cpd00067).
- `find_dup(...)` → returns either an `id_match` or `stoich_match` hit against the target model.
- `parse_equation(eq_str, compartment)` → builds a stoichiometry dict from the Excel `equation` column (supports `=>`, `<=>`, `<=` directions, `(N)` coefficients, optional `[c0]` compartment override).
- `get_or_create_met(...)` → resolves metabolites by lookup in fsp237 first, falling back to iMM904, then creating minimal stubs.

In [ ]:
COFACTORS_BASE = {'cpd00067'}  # H+
EQ_TERM_RE = re.compile(r'^\s*(?:\(?\s*(\d+(?:\.\d+)?)\s*\)?\s+)?(cpd\d+)(?:\[([a-z]\d+)\])?\s*$')

def canon_stoich(rxn):
    items = []
    for met, coef in rxn.metabolites.items():
        if met.id.split('_')[0] in COFACTORS_BASE: continue
        items.append((met.id, coef))
    return tuple(sorted(items))

def is_modelseed_id(rid):
    return bool(re.match(r'^rxn\d+', rid))

def build_stoich_index(model):
    idx = {}
    for r in model.reactions:
        idx.setdefault(canon_stoich(r), []).append(r.id)
    return idx

imm_by_id = {r.id: r for r in imm904.reactions}
imm_met_by_id = {m.id: m for m in imm904.metabolites}

def parse_equation(eq_str, rxn_compartment):
    if not isinstance(eq_str, str) or not eq_str.strip():
        return None, None
    eq = eq_str.strip()
    if '<=>' in eq:
        lhs, rhs = eq.split('<=>', 1); bounds = (-1000.0, 1000.0)
    elif '=>' in eq:
        lhs, rhs = eq.split('=>', 1); bounds = (0.0, 1000.0)
    elif '<=' in eq:
        rhs, lhs = eq.split('<=', 1); bounds = (0.0, 1000.0)
    else:
        return None, None
    stoich = {}
    for side, sign in [(lhs, -1), (rhs, +1)]:
        for term in side.split('+'):
            t = term.strip()
            if not t: continue
            m = EQ_TERM_RE.match(t)
            if not m: return None, None
            coef = float(m.group(1)) if m.group(1) else 1.0
            cpd  = m.group(2)
            cmpt = m.group(3) or rxn_compartment
            met_id = f'{cpd}_{cmpt}'
            stoich[met_id] = stoich.get(met_id, 0) + sign * coef
    return stoich, bounds

def get_or_create_met(model, met_id):
    if met_id in [m.id for m in model.metabolites]:
        return model.metabolites.get_by_id(met_id)
    if met_id in imm_met_by_id:
        m = imm_met_by_id[met_id].copy()
        model.add_metabolites([m])
        return m
    base = met_id.split('_')[0]
    cmpt = met_id.split('_')[-1]
    proto = next((imm_met_by_id[k] for k in imm_met_by_id if k.startswith(base + '_')), None)
    if proto:
        m = Metabolite(met_id, formula=proto.formula, name=proto.name.rsplit('_', 1)[0] + f'_{cmpt}', compartment=cmpt)
    else:
        m = Metabolite(met_id, compartment=cmpt)
    model.add_metabolites([m])
    return m

## Step 4 — Add Excel reactions to fsp237

Policy per Excel row:
1. **ID match in fsp237** → skip (`skipped_id_match`).
2. **Source the reaction**: prefer iMM904 by exact ID; fall back to iMM904 with stripped compartment; finally parse the `equation` column.
3. **Stoichiometric duplicate in fsp237** → if existing is BiGG-style and incoming is ModelSEED, *replace*; else keep existing.
4. **No duplicate** → add. Log the action.

In [ ]:
stoich_idx = build_stoich_index(fsp237)
a_counts = {'added_from_imm904':0, 'added_from_equation':0, 'skipped_id_match':0,
            'skipped_stoich_match':0, 'replaced_bigg_with_modelseed':0,
            'add_failed':0, 'parse_failed':0, 'compartment_remap':0}

for _, row in excel_df.iterrows():
    rid   = row['rxn id']
    sheet = row.get('source_sheet','?')
    eq    = row.get('equation')
    cmpt_match = re.search(r'_([a-z]\d+)$', rid)
    rxn_cmpt = cmpt_match.group(1) if cmpt_match else 'c0'

    if rid in [r.id for r in fsp237.reactions]:
        a_counts['skipped_id_match'] += 1
        build_log.append({'step':'A_excel','rxn_id':rid,'action':'skipped_id_match','detail':'already_in_fsp237'})
        continue

    src_kind, candidate = None, None
    if rid in imm_by_id:
        candidate = imm_by_id[rid].copy(); src_kind = 'imm904'
    else:
        base = re.sub(r'_[a-z]\d+$', '', rid)
        alt = next((r for r in imm904.reactions if r.id.startswith(base + '_')), None)
        if alt:
            candidate = alt.copy(); candidate.id = rid; src_kind = 'imm904_compartment_remap'
            a_counts['compartment_remap'] += 1
        elif isinstance(eq, str) and eq.strip():
            stoich, bounds = parse_equation(eq, rxn_cmpt)
            if stoich is None:
                a_counts['parse_failed'] += 1
                build_log.append({'step':'A_excel','rxn_id':rid,'action':'parse_failed','detail':f'eq={eq[:80]}'})
                continue
            new_rxn = Reaction(rid)
            new_rxn.name = str(row.get('name (ModelSEED)',''))[:120]
            new_rxn.lower_bound, new_rxn.upper_bound = bounds
            new_rxn.add_metabolites({get_or_create_met(fsp237, mid): coef for mid, coef in stoich.items()})
            candidate = new_rxn; src_kind = 'equation'
        else:
            a_counts['parse_failed'] += 1
            build_log.append({'step':'A_excel','rxn_id':rid,'action':'no_source_no_equation','detail':f'sheet={sheet}'})
            continue

    norm = canon_stoich(candidate)
    if norm in stoich_idx:
        existing_id = stoich_idx[norm][0]
        if (not is_modelseed_id(existing_id)) and is_modelseed_id(rid):
            try:
                fsp237.remove_reactions([existing_id])
                fsp237.add_reactions([candidate])
                stoich_idx = build_stoich_index(fsp237)
                a_counts['replaced_bigg_with_modelseed'] += 1
                build_log.append({'step':'A_excel','rxn_id':rid,'action':'replaced_bigg_with_modelseed',
                                  'detail':f'{existing_id} -> {rid} (src={src_kind})'})
            except Exception as e:
                build_log.append({'step':'A_excel','rxn_id':rid,'action':'replace_failed',
                                  'detail':f'{existing_id} -> {rid}; {str(e)[:120]}'})
        else:
            a_counts['skipped_stoich_match'] += 1
            build_log.append({'step':'A_excel','rxn_id':rid,'action':'skipped_stoich_match',
                              'detail':f'existing={existing_id}'})
        continue

    try:
        fsp237.add_reactions([candidate])
        stoich_idx.setdefault(canon_stoich(candidate), []).append(rid)
        if src_kind == 'equation':
            a_counts['added_from_equation'] += 1
        else:
            a_counts['added_from_imm904'] += 1
        build_log.append({'step':'A_excel','rxn_id':rid,'action':f'added_{src_kind}','detail':''})
    except Exception as e:
        a_counts['add_failed'] += 1
        build_log.append({'step':'A_excel','rxn_id':rid,'action':'add_failed','detail':str(e)[:120]})

print('Step A counts:', ', '.join(f'{k}={v}' for k, v in a_counts.items()))
print(f'fsp237 after Excel: {len(fsp237.reactions)} rxns, {len(fsp237.metabolites)} mets')

## Step 5 — Apply GMM media + glucose uptake = 5

Pulls the GMM media object from KBase (`19994/361/1`), resets all exchange reactions, then applies each media constraint that has a matching exchange in the model. Finally overrides glucose exchange uptake to `-5` mmol/gDW/h (more realistic than the default `-1`).

In [ ]:
media = kbase.get_from_ws('19994/361/1')
media_constraints = media.get_media_constraints()
print(f'GMM constraints: {len(media_constraints)} entries')

for ex in fsp237.exchanges:
    if ex.upper_bound < 0: ex.upper_bound = 0
    ex.lower_bound = 0

applied = 0; not_in_model = []
for cpd_id, (lb, ub) in media_constraints.items():
    ex_id = f'EX_{cpd_id}'
    if ex_id in [r.id for r in fsp237.exchanges]:
        fsp237.reactions.get_by_id(ex_id).lower_bound = lb
        fsp237.reactions.get_by_id(ex_id).upper_bound = ub
        applied += 1
    else:
        not_in_model.append(cpd_id)
print(f'applied {applied}/{len(media_constraints)} (missing exchanges: {not_in_model})')

fsp237.reactions.EX_cpd00027_e0.lower_bound = -GLC_UPTAKE
print(f'EX_cpd00027_e0 (glc) lower_bound = -{GLC_UPTAKE}')

fsp237.objective = 'bio1'
sol_pre = fsp237.optimize()
print(f'biomass (fsp237 + Excel, no iMM904 essentials): {sol_pre.objective_value or 0:.6g}')

## Step 6 — Greedy minimal iMM904 essentials

1. Identify iMM904 candidates not already in fsp237 (by ID or stoichiometry).
2. Add all candidates → "super-model". Confirm biomass grows.
3. KO each candidate in random order: if biomass stays ≥ 95% of super-model, mark non-essential. Otherwise restore and keep.
4. Remove all non-essentials at the end. Result is the minimum iMM904 set required for biomass.

In [ ]:
fsp_ids_now = set(r.id for r in fsp237.reactions)
fsp_stoich  = build_stoich_index(fsp237)
candidates = []
for r in imm904.reactions:
    if r.id in fsp_ids_now: continue
    if canon_stoich(r) in fsp_stoich: continue
    candidates.append(r.id)
print(f'iMM904 candidates not already in fsp237: {len(candidates)}')

for rid in candidates:
    try:
        fsp237.add_reactions([imm_by_id[rid].copy()])
    except Exception as e:
        build_log.append({'step':'C_imm_add_super','rxn_id':rid,'action':'add_failed','detail':str(e)[:120]})

sol_super = fsp237.optimize()
g_super = sol_super.objective_value or 0
threshold = max(EPS_BIO, g_super * BIO_THRESH_FRAC)
print(f'super-model biomass: {g_super:.6g}  ({len(fsp237.reactions)} rxns)')
print(f'KO threshold (biomass >= {threshold:.6g})')

kept_essential = []
if g_super >= EPS_BIO:
    actually_in = [rid for rid in candidates if rid in [r.id for r in fsp237.reactions]]
    random.shuffle(actually_in)
    t0 = time.time()
    for rid in actually_in:
        rxn = fsp237.reactions.get_by_id(rid)
        old_lb, old_ub = rxn.lower_bound, rxn.upper_bound
        rxn.lower_bound = 0; rxn.upper_bound = 0
        sol = fsp237.optimize()
        bio = sol.objective_value or 0
        if bio >= threshold:
            build_log.append({'step':'C_imm_ko','rxn_id':rid,'action':'dropped_non_essential',
                              'detail':f'bio_after_ko={bio:.6g}'})
        else:
            rxn.lower_bound = old_lb; rxn.upper_bound = old_ub
            kept_essential.append(rid)
            build_log.append({'step':'C_imm_ko','rxn_id':rid,'action':'kept_essential',
                              'detail':f'bio_after_ko={bio:.6g}'})
    print(f'KO-prune done in {time.time()-t0:.1f}s')

to_remove = [rid for rid in candidates if rid in [r.id for r in fsp237.reactions] and rid not in kept_essential]
fsp237.remove_reactions(to_remove)
print(f'removed {len(to_remove)} non-essentials')
print(f'kept {len(kept_essential)} essentials: {kept_essential}')

sol_final = fsp237.optimize()
print(f'\nFINAL biomass: {sol_final.objective_value or 0:.6g}')
print(f'FINAL model: {len(fsp237.reactions)} rxns, {len(fsp237.metabolites)} mets')

## Step 6.5 — Final dedup pass (lower-MS-wins)

Step 4's add logic only handles one direction of duplication (BiGG-style → ModelSEED swap). After Steps 4–6 there can still be:

- **MS-vs-MS duplicates** where the higher-numbered ModelSEED ID was already in the fsp237 baseline and the lower one came in from Excel/iMM904.
- **Compartment-remap leftovers** where the equation parser kept c0 metabolites in an `_m0` reaction shell — same chemistry as the c0 version.
- **iMM904 essentials** that overlap with reactions Excel added earlier.

This pass groups every reaction by canonical stoichiometry (compartment-aware, direction-aware, H+ ignored) and for each duplicate group:

1. If any ModelSEED IDs are present → keep the **lowest numeric suffix**, drop the rest (MS or non-MS).
2. Otherwise → keep the first, drop the rest.

The kept reaction inherits the loser's GPR if it had none, and its bounds are widened to the union of the two — so we never silently constrain flux. Orphan metabolites are pruned at the end. Every removal is logged to `fsp237_dedup_removed_log.tsv`.

In [ ]:
from collections import defaultdict

MS_RE = re.compile(r'^rxn(\d+)')
def _ms_num(rid):
    mtch = MS_RE.match(rid)
    return int(mtch.group(1)) if mtch else None

dup_groups = defaultdict(list)
for r in fsp237.reactions:
    k = canon_stoich(r)
    if not k:
        continue
    dup_groups[k].append(r.id)
dup_groups = {k: v for k, v in dup_groups.items() if len(v) > 1}
print(f'duplicate groups found: {len(dup_groups)}')

dedup_log = []
for ids in dup_groups.values():
    ms_ids = sorted([r for r in ids if _ms_num(r) is not None], key=_ms_num)
    nms    = [r for r in ids if _ms_num(r) is None]
    if ms_ids:
        keep   = ms_ids[0]
        remove = ms_ids[1:] + nms
    else:
        keep   = ids[0]
        remove = ids[1:]
    kp = fsp237.reactions.get_by_id(keep)
    for rid in remove:
        rm = fsp237.reactions.get_by_id(rid)
        merged_gpr = False
        if not kp.gene_reaction_rule and rm.gene_reaction_rule:
            kp.gene_reaction_rule = rm.gene_reaction_rule
            merged_gpr = True
        new_lb = min(kp.lower_bound, rm.lower_bound)
        new_ub = max(kp.upper_bound, rm.upper_bound)
        widened = (new_lb, new_ub) != (kp.lower_bound, kp.upper_bound)
        if widened:
            kp.lower_bound, kp.upper_bound = new_lb, new_ub
        dedup_log.append({
            'removed_rxn'  : rid,
            'kept_rxn'     : keep,
            'removed_name' : rm.name,
            'kept_name'    : kp.name,
            'removed_eq'   : rm.build_reaction_string(),
            'kept_eq'      : kp.build_reaction_string(),
            'removed_lb'   : rm.lower_bound,
            'removed_ub'   : rm.upper_bound,
            'kept_lb'      : kp.lower_bound,
            'kept_ub'      : kp.upper_bound,
            'removed_genes': ';'.join(g.id for g in rm.genes),
            'kept_genes'   : ';'.join(g.id for g in kp.genes),
            'merged_gpr_from_removed': merged_gpr,
            'widened_bounds_to': f'[{new_lb},{new_ub}]' if widened else '',
        })
        build_log.append({'step':'D_dedup','rxn_id':rid,'action':'removed_duplicate',
                          'detail':f'kept={keep}'})

fsp237.remove_reactions([e['removed_rxn'] for e in dedup_log])
orphans = [met for met in fsp237.metabolites if len(met.reactions) == 0]
if orphans:
    fsp237.remove_metabolites(orphans)
print(f'removed {len(dedup_log)} duplicate reactions, {len(orphans)} orphan metabolites')

sol_dd = fsp237.optimize()
print(f'biomass after dedup: {sol_dd.objective_value or 0:.6g}  ({len(fsp237.reactions)} rxns, {len(fsp237.metabolites)} mets)')

pd.DataFrame(dedup_log).to_csv(dedup_tsv, sep='\t', index=False)
print(f'dedup log: {dedup_tsv} ({len(dedup_log)} entries)')

## Step 7 — Save outputs

- `fsp237_minimal_glucose.json` — the final model
- `fsp237_minimal_glucose_fluxes.tsv` — full flux vector for the FBA solution
- `fsp237_build_log.tsv` — per-reaction add/skip/replace/KO log

In [ ]:
sol_final = fsp237.optimize()
print(f'FINAL biomass (post-dedup): {sol_final.objective_value or 0:.6g}')
print(f'FINAL model: {len(fsp237.reactions)} rxns, {len(fsp237.metabolites)} mets')

cobra.io.save_json_model(fsp237, final_json)
sol_final.fluxes.to_csv(final_tsv, sep='\t', header=['flux'])
pd.DataFrame(build_log).to_csv(log_tsv, sep='\t', index=False)
print(f'model : {final_json}')
print(f'fluxes: {final_tsv}')
print(f'log   : {log_tsv}  ({len(build_log)} entries)')

## Step 8 — Compartment-coloured Escher flux map

Reuses the colour palette established in the diagnostic notebook: cytosol pink, other compartments in distinct categorical hues, edge thickness and intensity scaled by flux magnitude. The kept iMM904 essentials are highlighted bright red so they stand out against the rest of the central-carbon flow.

In [ ]:
import re as _re


model_p  = final_json
out_html = map_html


m = cobra.io.load_json_model(model_p)
sol = m.optimize()
g = sol.objective_value or 0
print(f'biomass: {g:.6g}')
active = sol.fluxes[sol.fluxes.abs() > 1e-6]
flux_dict = active.to_dict()
print(f'active rxns (|flux|>1e-6): {len(active)}')
print(f'  rxn00558_c0 (PGI)   : {sol.fluxes.get("rxn00558_c0", "absent"):+.6g}')
print(f'  rxn00604_c0 (G6PDH) : {sol.fluxes.get("rxn00604_c0", "absent"):+.6g}')

with open(escher_map_path) as f:
    map_json_str = f.read()
map_data = json.loads(map_json_str)
map_rxn_ids = {r['bigg_id'] for r in map_data[1]['reactions'].values()}
print(f'active on central-carbon map: {sum(1 for r in flux_dict if r in map_rxn_ids)} / {len(map_rxn_ids)}')

COMPARTMENT_COLORS = {
    'c0': ('#ff69b4', '#ff1493'),
    'r0': ('#aec7e8', '#1f77b4'),
    'm0': ('#98df8a', '#2ca02c'),
    'x0': ('#ffbb78', '#ff7f0e'),
    'e0': ('#c49c94', '#8c564b'),
    'n0': ('#9edae5', '#17becf'),
    'g0': ('#dbdb8d', '#bcbd22'),
    'v0': ('#c5b0d5', '#9467bd'),
}
DEFAULT_COLORS = ('#cc0000', '#cc0000')
IMM_ESSENTIAL_RED = '#cc0000'
IMM_ESSENTIALS = set(kept_essential) if 'kept_essential' in dir() else {'rxn08475_c0', 'rxn00881_c0', 'ORNt3m_c0'}

def _hex_interp(low, high, t):
    h2r = lambda h: tuple(int(h.lstrip('#')[i:i+2], 16) for i in (0, 2, 4))
    r1, g1, b1 = h2r(low); r2, g2, b2 = h2r(high)
    return '#{:02x}{:02x}{:02x}'.format(
        int(r1 + (r2 - r1) * t),
        int(g1 + (g2 - g1) * t),
        int(b1 + (b2 - b1) * t),
    )

max_abs = max((abs(v) for v in flux_dict.values()), default=1.0) or 1.0
color_map, size_map = {}, {}
for rid, flux in flux_dict.items():
    if abs(flux) <= 1e-9: continue
    try:
        rxn = m.reactions.get_by_id(rid)
    except KeyError:
        continue
    suffixes = [met.id.rsplit('_', 1)[-1] for met in rxn.metabolites if '_' in met.id]
    comp = max(set(suffixes), key=suffixes.count) if suffixes else 'c0'
    t = 0.35 + 0.65 * (abs(flux) / max_abs) ** 0.35
    if rid in IMM_ESSENTIALS:
        color_map[rid] = IMM_ESSENTIAL_RED
    else:
        low, high = COMPARTMENT_COLORS.get(comp, DEFAULT_COLORS)
        color_map[rid] = _hex_interp(low, high, t)
    size_map[rid] = 3 + t * 18

builder = Builder(
    map_json=map_json_str,
    reaction_data=flux_dict,
    reaction_scale=[
        {'type': 'min',    'color': '#cccccc', 'size': 2},
        {'type': 'value',  'value': 0, 'color': '#cccccc', 'size': 2},
        {'type': 'median', 'color': '#888888', 'size': 10},
        {'type': 'max',    'color': '#111111', 'size': 20},
    ],
    reaction_no_data_color='#dcdcdc',
    reaction_no_data_size=3,
)
builder.save_html(out_html)

with open(out_html) as f:
    html = f.read()
html = _re.sub(r'<script>\s*\(function\(\).*?\}\)\(\);\s*</script>', '', html, flags=_re.DOTALL)
html = _re.sub(r'<div id="compartment-legend".*?</div>\s*', '', html, flags=_re.DOTALL)

color_js = json.dumps(color_map); size_js = json.dumps(size_map)
legend_items = [
    ('#ff1493', 'Cytosol (c0)'),
    ('#1f77b4', 'ER (r0)'),
    ('#2ca02c', 'Mitochondria (m0)'),
    ('#ff7f0e', 'Peroxisome (x0)'),
    ('#8c564b', 'Extracellular (e0)'),
    ('#17becf', 'Nucleus (n0)'),
    ('#bcbd22', 'Golgi (g0)'),
    ('#9467bd', 'Vacuole (v0)'),
    ('#cc0000', 'iMM904 essentials (kept)'),
    ('#dcdcdc', 'No flux / no data'),
]
legend_html = ''.join(
    f'<div style="display:flex;align-items:center;margin-bottom:5px">'
    f'<div style="width:26px;height:4px;background:{c};margin-right:8px;border-radius:2px"></div>'
    f'<span>{lbl}</span></div>'
    for c, lbl in legend_items
)

inject = f"""
<script>
(function() {{
  var colorMap = {color_js};
  var sizeMap  = {size_js};
  function applyColors() {{
    var applied = 0;
    document.querySelectorAll('g.reaction').forEach(function(g) {{
      var labelEl = g.querySelector('.reaction-label');
      if (!labelEl) return;
      var rxnId = labelEl.textContent.trim().split(/\\s+/)[0];
      if (colorMap[rxnId]) {{
        var color = colorMap[rxnId];
        var size  = (sizeMap[rxnId] || 2) + 'px';
        g.querySelectorAll('path, line, polyline').forEach(function(p) {{
          p.style.stroke = color;
          p.style.strokeWidth = size;
        }});
        labelEl.style.fill = color;
        applied++;
      }}
    }});
    console.log('Compartment colors applied to', applied, 'active reactions');
  }}
  var attempts = 0;
  var interval = setInterval(function() {{
    if (document.querySelectorAll('g.reaction').length > 0) {{
      clearInterval(interval);
      applyColors();
    }} else if (++attempts > 60) {{
      clearInterval(interval);
    }}
  }}, 200);
}})();
</script>

<div id="compartment-legend" style="
  position:fixed; bottom:20px; right:20px;
  background:rgba(255,255,255,0.97);
  border:1px solid #ddd; border-radius:10px;
  padding:16px 20px; font-family:sans-serif;
  font-size:12px; z-index:9999;
  box-shadow:0 3px 12px rgba(0,0,0,0.18)">
  <b style="font-size:13px">fsp237 minimal model &mdash; glc uptake = 5</b>
  <div style="margin:8px 0">{legend_html}</div>
  <i style="color:#999;font-size:11px">Intensity &amp; thickness = flux magnitude</i><br>
  <i style="color:#999;font-size:11px">fsp237 + all Excel rxns + 3 iMM904 essentials &middot; biomass={g:.4f}</i>
</div>
"""

html = html.replace('</body>', inject + '\n</body>')
with open(out_html, 'w') as f:
    f.write(html)
print(f'wrote {out_html} ({os.path.getsize(out_html)/1024:.1f} KB)')
